# Experimental RNN augmentation workflow

This notebook is a later experimental augmentation workflow and is **not** the final preprocessing implementation reported in the manuscript. In this file, the BiLSTM is fitted before the final group-wise split. The manuscript workflow instead performed the group-wise split first and excluded validation groups from BiLSTM fitting. The original split-first notebook was overwritten during later development.


#### RNN Data Augmentation Model

This notebook implements a data augmentation pipeline using a Bidirectional LSTM model to generate synthetic training samples for rare data regions.

**Workflow:**
1. **Data Loading and Preprocessing**: Load training data and standardize features
2. **RNN Model Training**: Train BiLSTM model on full dataset 
3. **Data Augmentation**: Augment full dataset with all material groups participating
4. **Group-based Split**: Use GroupShuffleSplit for train/val split to prevent data leakage
5. **Visualization**: Compare original vs augmented data distributions

**Augmentation Principles:**
- **Preserved Features**: 'Ep1', 'Ep2', 'Ep3', 'Epp1', 'Epp2', 'Epp3' - Copied exactly from rare data
- **Perturbed Features**: 't1', 't2' - Small Gaussian noise added (5% of standard deviation)
- **RNN-Predicted Features**: 'EpH', 'EppH' - Predicted by RNN model based on perturbed sequences

**Scaler Handling:**
- `scaler_x.pkl` and `scaler_y.pkl`: Fitted on full original data, used for RNN training and data augmentation
- `scaler_x_new.pkl` and `scaler_y_new.pkl`: Refitted on augmented training set, recommended for diffusion model

**Advantages:**
- Diffusion model can learn from all material groups as conditions
- GroupShuffleSplit ensures samples from the same material group stay together
- More complete coverage of material parameter space

**Required Modules:**
`numpy`, `pandas`, `scikit-learn`, `torch`, `matplotlib`


#### Environment Settings

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import math
import copy
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from torch.optim.swa_utils import AveragedModel
from matplotlib.ticker import MultipleLocator
from scipy.stats import gaussian_kde
import random

# Set random seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Set matplotlib font
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['DejaVu Serif', 'Liberation Serif']
plt.rcParams['font.weight'] = 'bold'

#### 1. Data Loading and Preprocessing

In [ ]:
# Configuration parameters
DATA_PATH = "data/TrainingData.xlsx"
SEQ_LEN = 10  # Sequence length
AUG_FACTOR = 4  # Augmentation factor
RARE_BOUNDS_RAW = np.array([[1.20, 0.20], [5.80, 2.20]], dtype=np.float32)  # Rare data bounds

# Define column names
X_cols = ['Ep1', 'Ep2', 'Ep3', 'Epp1', 'Epp2', 'Epp3', 't1', 't2']  # Input features
y_cols = ['EpH', 'EppH']  # Output targets
MOD = ['Ep1', 'Ep2', 'Ep3', 'Epp1', 'Epp2', 'Epp3']  # Modulus columns (for grouping)

print("Configuration parameters set")

In [ ]:
def ensure_expected_cols(df):
    """Ensure dataframe contains expected columns"""
    expect = X_cols + y_cols
    missing = [c for c in expect if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    return df[expect].copy()

# Load data
df = pd.read_excel(DATA_PATH)
df = ensure_expected_cols(df).reset_index(drop=True)
print(f"Data loaded successfully, {len(df)} rows")

# Display target variable statistics
y_df = df[y_cols]
print("\nTarget variable statistics:")
print(y_df.describe())

for col in y_cols:
    col_min = y_df[col].min()
    col_max = y_df[col].max()
    print(f"{col}: min = {col_min:.4f}, max = {col_max:.4f}")

In [ ]:
# ===== Fit scalers on full dataset (for RNN training and augmentation) =====

N = len(df)
groups = pd.util.hash_pandas_object(df[MOD].round(6), index=False).astype(np.int64).to_numpy()
print(f"Total rows: {N}, unique groups: {len(np.unique(groups))}")

# Fit scalers on full dataset
sx = StandardScaler().fit(df[X_cols].to_numpy(float))
sy = StandardScaler().fit(df[y_cols].to_numpy(float))

import os, joblib
os.makedirs("artifacts", exist_ok=True)
joblib.dump(sx, "artifacts/experimental/scaler_x.pkl")
joblib.dump(sy, "artifacts/experimental/scaler_y.pkl")
print("[RNN] Saved scalers (based on full data) to artifacts/experimental/scaler_x.pkl & artifacts/experimental/scaler_y.pkl")

print("Data standardization completed (using full dataset)")

In [ ]:
def make_windows_from_df(df, seq_len=SEQ_LEN):
    """Convert data to sequence windows"""
    X_scaled = sx.transform(df[X_cols].to_numpy(float))
    y_scaled = sy.transform(df[y_cols].to_numpy(float))
    Xy = np.hstack([X_scaled, y_scaled]).astype(np.float32)
    N = len(Xy)
    
    if N < seq_len:
        raise ValueError(f"Data rows {N} < sequence length {seq_len}")
    
    X_list, y_list = [], []
    for i in range(N - seq_len + 1):
        X_list.append(Xy[i:i+seq_len-1, :8])  # Input sequence
        y_list.append(Xy[i+seq_len-1, 8:10])  # Target values
    
    return np.asarray(X_list, np.float32), np.asarray(y_list, np.float32)

# Generate windows from full dataset (for RNN training)
X_all_w, y_all_w = make_windows_from_df(df, SEQ_LEN)

# Split into training and validation windows (for early stopping during RNN training)
Xtr_w_tr, Xtr_w_val, ytr_w_tr, ytr_w_val = train_test_split(
    X_all_w, y_all_w, test_size=0.15, random_state=SEED
)

print(f"Window data generation completed (full dataset): total windows {len(X_all_w)}")
print(f"For RNN training: train windows {len(Xtr_w_tr)}, validation windows {len(Xtr_w_val)}")

#### 2. Original Data Distribution Visualization


In [ ]:
# Plot target variable distribution histograms
def plot_histogram(data, title, xlabel, bins=30):
    """Plot histogram for a single variable"""
    plt.figure(figsize=(8, 6))
    
    # Get histogram data
    counts, bins_edges, patches = plt.hist(data, bins=bins, edgecolor='k', alpha=0.7)
    
    # Format y-axis to show values divided by 1000
    ax = plt.gca()
    
    # Define formatter function to divide by 1000
    def format_y_ticks(x, pos):
        return f'{x/1000:.0f}'
    
    # Apply the formatter
    from matplotlib.ticker import FuncFormatter
    ax.yaxis.set_major_formatter(FuncFormatter(format_y_ticks))
    
    plt.xlabel(xlabel, fontsize=24, fontweight='bold', fontfamily='serif', labelpad=10)
    plt.ylabel(r'Frequency ($\times 10^3$)', fontsize=24, fontweight='bold', fontfamily='serif', labelpad=8)
    plt.xticks(fontsize=20, fontweight='bold', fontfamily='serif')
    plt.yticks(fontsize=20, fontweight='bold', fontfamily='serif')
    
    ax.xaxis.set_major_locator(MultipleLocator(base=1))
    ax.yaxis.set_major_locator(MultipleLocator(base=1000))
    
    for spine in ax.spines.values():
        spine.set_linewidth(3)
    plt.tick_params(axis='both', which='major', length=8, width=3)
    plt.tight_layout()
    plt.show()

# Plot EpH distribution
plot_histogram(y_df['EpH'], r'$\mathbf{E_H^{\prime}}$ (MPa)', r'$E_H^{\prime}$ (MPa)')

# Plot EppH distribution
plot_histogram(y_df['EppH'], r'$\mathbf{E_H^{\prime\prime}}$ (MPa)', r'$E_H^{\prime\prime}$ (MPa)')


#### 3. RNN Model Definition and Training


In [ ]:
# Calculate frequency distribution required for Focal weights
# Use full dataset to calculate frequency distribution
_y_train_raw = df[y_cols].to_numpy(float)
bins_x, bins_y = 40, 40
eps = 1e-8
H, xedges, yedges = np.histogram2d(_y_train_raw[:,0], _y_train_raw[:,1],
                                   bins=(bins_x, bins_y), density=True)
H = H + eps  # Prevent division by zero
H_max = H.max()

# Focal weight parameters
FOCAL_GAMMA = 1.0
ALPHA_MAX = 3.0

def batch_focal_weights(yb_scaled):
    """Calculate Focal weights, biased towards rare data regions"""
    yb_raw = sy.inverse_transform(yb_scaled.detach().cpu().numpy())
    # Find the 2D bin for each sample
    ix = np.clip(np.digitize(yb_raw[:,0], xedges) - 1, 0, H.shape[0]-1)
    iy = np.clip(np.digitize(yb_raw[:,1], yedges) - 1, 0, H.shape[1]-1)
    freq = H[ix, iy]  # Lower frequency means more rare
    # Frequency → weight: w = (H_max / freq)^gamma, limited to [1, ALPHA_MAX]
    w = np.clip((H_max / freq) ** FOCAL_GAMMA, 1.0, ALPHA_MAX).astype('float32')
    return torch.from_numpy(w)

print("Focal weight calculation function defined")

In [ ]:
class BiLSTMRegressor(nn.Module):
    """Bidirectional LSTM regression model"""
    def __init__(self, input_dim=8, hidden=64, num_layers=2, dropout=0.0, out_dim=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, num_layers=num_layers,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden*2),
            nn.Linear(hidden*2, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, out_dim),
        )
    
    def forward(self, x):  # x: (B,T,8)
        # Ensure weights are contiguous on CUDA
        if x.is_cuda:
            try:
                self.lstm.flatten_parameters()
            except RuntimeError:
                pass
        
        out, _ = self.lstm(x)
        last = out[:, -1, :]  # Take output from last time step
        return self.head(last)

def weighted_huber(pred, target, w, delta=0.5):
    """Weighted Huber loss function"""
    err = pred - target
    abs_e = torch.abs(err)
    quad = torch.minimum(abs_e, torch.tensor(delta, device=err.device))
    lin = abs_e - quad
    loss = 0.5 * quad**2 + delta * lin  # (B,2)
    return (w * loss.sum(dim=1)).mean()

print("BiLSTM model and loss function defined")

In [ ]:
def train_rnn(X_train, y_train, X_val, y_val,
              epochs=80, bs=512, base_lr=2e-3, wd=1e-4, device=DEVICE,
              warmup_epochs=8, patience_init=8, min_delta=1e-3,
              focal_start_epoch=10, focal_full_epoch=50,
              ema_decay=0.995, noise_scale=0.005):
    """Train RNN model"""
    
    # Model and optimizer setup
    model = BiLSTMRegressor().to(device)
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if p.ndimension() == 1 or 'bias' in n.lower() or 'norm' in n.lower():
            no_decay.append(p)
        else:
            decay.append(p)
    
    opt = torch.optim.AdamW(
        [{'params': decay, 'weight_decay': wd},
         {'params': no_decay, 'weight_decay': 0.0}],
        lr=base_lr
    )
    
    # Learning rate scheduler: warmup + cosine
    def lr_lambda(ep):
        if ep < warmup_epochs:
            return (ep + 1) / warmup_epochs
        t = (ep - warmup_epochs) / max(1, (epochs - warmup_epochs))
        return 0.5 * (1.0 + math.cos(math.pi * t))
    
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    
    # EMA model
    ema_model = AveragedModel(model, avg_fn=lambda avg, cur, n: avg*ema_decay + cur*(1-ema_decay))
    
    # Data loaders
    tr_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    va_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
    tr_dl = DataLoader(tr_ds, batch_size=bs, shuffle=True)
    va_dl = DataLoader(va_ds, batch_size=bs, shuffle=False)
    
    mse = nn.MSELoss(reduction="mean")
    hist = {"epoch": [], "train_loss": [], "val_loss": [], "val_mse": [], "lr": []}
    
    best = float('inf')
    best_state = None
    patience = patience_init
    from collections import deque
    smooth = deque(maxlen=3)  # Validation MSE moving average
    
    print("Starting RNN model training...")
    
    for ep in range(1, epochs+1):
        # Training phase
        model.train()
        tr_sum = 0.0
        n_tr = 0
        
        for xb, yb in tr_dl:
            xb, yb = xb.to(device), yb.to(device)
            
            # Add small input noise
            with torch.no_grad():
                estd = xb.std(dim=(0,1), keepdim=True) + 1e-3
            scale = noise_scale * (0.5 if ep < warmup_epochs else 1.0)
            if ep >= 40:
                scale *= 0.5
            xb = xb + torch.randn_like(xb) * (scale * estd)
            
            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            
            # Focal weight linear transition
            if ep < focal_start_epoch:
                w = torch.ones(len(xb), device=xb.device)
            else:
                alpha = min(1.0, (ep - focal_start_epoch) / max(1, focal_full_epoch - focal_start_epoch))
                w_raw = batch_focal_weights(yb).to(device)
                w = 1.0 + alpha * (w_raw - 1.0)
            
            loss = weighted_huber(pred, yb, w, delta=0.25)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            
            # Update EMA model
            ema_model.update_parameters(model)
            
            tr_sum += loss.item() * len(xb)
            n_tr += len(xb)
        
        tr_loss = tr_sum / max(1, n_tr)
        
        # Validation phase (using EMA model)
        ema_model.eval()
        va_mse_sum = 0.0
        n_va = 0
        
        with torch.no_grad():
            for xb, yb in va_dl:
                xb, yb = xb.to(device), yb.to(device)
                pred = ema_model(xb)
                va_mse_sum += mse(pred, yb).item() * len(xb)
                n_va += len(xb)
        
        va_mse = va_mse_sum / max(1, n_va)
        
        # Record history
        hist["epoch"].append(ep)
        hist["train_loss"].append(tr_loss)
        hist["val_loss"].append(va_mse)
        hist["val_mse"].append(va_mse)
        hist["lr"].append(opt.param_groups[0]["lr"])
        
        if ep % 10 == 0 or ep <= 10:
            print(f"Epoch {ep:3d} | train {tr_loss:.4f} | val(mse-EMA) {va_mse:.4f}")
        
        sch.step()
        
        # Early stopping logic
        smooth.append(va_mse)
        monitor = sum(smooth)/len(smooth)
        improved = (best - monitor) > min_delta
        
        if ep <= warmup_epochs:
            if monitor < best:
                best = monitor
                best_state = copy.deepcopy(ema_model.state_dict())
            patience = patience_init
            continue
        
        if improved:
            best = monitor
            best_state = copy.deepcopy(ema_model.state_dict())
            patience = patience_init
        else:
            patience -= 1
            if patience <= 0:
                print(f"Early stopping at epoch {ep} (best≈{best:.4f})")
                best_ep = int(np.argmin(hist["val_mse"])) + 1
                print(f"[Best] epoch={best_ep}, val_mse={min(hist['val_mse']):.4f}")
                break
    
    # Load best weights
    if best_state is not None:
        ema_model.load_state_dict(best_state)
    model.load_state_dict(ema_model.state_dict(), strict=False)
    
    print("RNN model training completed")
    return model, hist

# Train model
model_rnn, hist = train_rnn(Xtr_w_tr, ytr_w_tr, Xtr_w_val, ytr_w_val,
                           epochs=80, patience_init=10, warmup_epochs=10,
                           min_delta=8e-4, focal_start_epoch=10, noise_scale=0.005)

#### 4. Data Augmentation


In [ ]:
@torch.no_grad()
def batched_predict(model, X_np, bs=512, device=DEVICE):
    """Batch prediction"""
    model.eval()
    outs = []
    for i in range(0, len(X_np), bs):
        xb = torch.from_numpy(X_np[i:i+bs]).to(device)
        outs.append(model(xb).cpu().numpy().astype(np.float32))
    return np.vstack(outs)

def augment_train_side(df_input, sx, sy, model, seq_len=SEQ_LEN, aug_factor=AUG_FACTOR, rare_bounds_raw=RARE_BOUNDS_RAW):
    """
    Augment training data
    - Preserve Ep1, Ep2, Ep3, Epp1, Epp2, Epp3 unchanged
    - Add perturbations to t1, t2
    - Use RNN to predict EpH, EppH
    """
    X_raw_np = df_input[X_cols].to_numpy(float)
    y_raw_np = df_input[y_cols].to_numpy(float)
    X_scaled = sx.transform(X_raw_np)
    y_scaled = sy.transform(y_raw_np)
    Xy = np.hstack([X_scaled, y_scaled]).astype(np.float32)
    
    N = len(Xy)
    Nw = N - (seq_len - 1)
    
    if Nw <= 0:
        print(f"[Augment] Data rows {N} < sequence length {seq_len} → skip")
        return df_input.copy()
    
    # Find rare data windows
    Y_last = np.stack([Xy[i+(seq_len-1), 8:10] for i in range(Nw)], 0)
    low_s, high_s = sy.transform(rare_bounds_raw)[0], sy.transform(rare_bounds_raw)[1]
    mask = ((Y_last[:,0] >= low_s[0]) & (Y_last[:,0] <= high_s[0]) & 
            (Y_last[:,1] >= low_s[1]) & (Y_last[:,1] <= high_s[1]))
    rare_starts = np.where(mask)[0].astype(int)
    
    print(f"[Augment] Rare data windows: {len(rare_starts)}")
    
    if len(rare_starts) == 0 or aug_factor <= 1:
        print("[Augment] Skip")
        return df_input.copy()
    
    # Generate synthetic data
    num_syn = len(rare_starts) * (aug_factor - 1)
    starts = np.random.choice(rare_starts, size=num_syn, replace=True).astype(int)
    
    idx_mat = starts[:,None] + np.arange(seq_len-1)[None,:]
    if idx_mat.max() >= N:
        raise IndexError(f"Index out of range: idx_mat.max={idx_mat.max()} >= N={N}")
    
    # === Add perturbations to t1, t2 ===
    X_in = Xy[idx_mat, :8].copy()  # (K, seq_len-1, 8)
    
    # Add small Gaussian noise to t1, t2 (indices 6, 7)
    t_cols_idx = [6, 7]  # t1, t2 indices in X_cols
    for t_idx in t_cols_idx:
        # Calculate feature standard deviation (in scaled space)
        t_std = X_in[:, :, t_idx].std()
        # Add noise (5% of standard deviation)
        noise = np.random.normal(0, 0.05 * t_std, size=X_in[:, :, t_idx].shape)
        X_in[:, :, t_idx] += noise.astype(np.float32)
    
    # Predict target values (based on perturbed sequences)
    yhat_scaled = batched_predict(model, X_in)
    yhat_raw = sy.inverse_transform(yhat_scaled)
    
    # Get input features for output
    last_idx = starts + (seq_len-1)
    if last_idx.max() >= N:
        raise IndexError(f"Index out of range: last_idx.max={last_idx.max()} >= N={N}")
    
    # === Construct output features: preserve Ep1-Epp3 (exact copy), use perturbed t1, t2 ===
    # Get Ep1-Epp3 directly from original DataFrame to avoid floating point precision issues
    X_mod_raw = df_input.iloc[last_idx][MOD].to_numpy(float).astype(np.float32)
    # Round to 4 decimal places to prevent floating point precision issues
    X_mod_raw = np.round(X_mod_raw, 4).astype(np.float32)
    
    # Get perturbed t1, t2 from sequences and inverse transform
    X_t_scaled = X_in[:, -1, 6:8]  # t1, t2 (perturbed values)
    X_t_raw = (X_t_scaled * sx.scale_[6:8] + sx.mean_[6:8]).astype(np.float32)
    
    # Combine all input features: [Ep1-Epp3(exact), t1-t2(perturbed)]
    X_last_raw = np.hstack([X_mod_raw, X_t_raw])
    
    # Combine synthetic data
    syn_np = np.hstack([X_last_raw, yhat_raw]).astype(np.float32)
    syn_df = pd.DataFrame(syn_np, columns=X_cols + y_cols)
    
    # Merge original and synthetic data
    df_aug = pd.concat([df_input, syn_df], ignore_index=True)
    
    # Fix floating point precision issues in 6-tuple columns
    for col in MOD:
        df_aug[col] = df_aug[col].round(4)
    
    print(f"[Augment] Added {len(syn_df)} rows → total size = {len(df_aug)}")
    print(f"[Augment] Preserved: {MOD}")
    print(f"[Augment] Perturbed: t1, t2")
    print(f"[Augment] RNN predicted: {y_cols}")
    
    return df_aug

# Execute data augmentation
df_aug = augment_train_side(df, sx, sy, model_rnn, seq_len=SEQ_LEN, aug_factor=AUG_FACTOR, rare_bounds_raw=RARE_BOUNDS_RAW)

In [ ]:
# ===== After augmentation, perform group-based train/val split =====
print("\n[GroupSplit] Performing group-based split on augmented data...")

# Group augmented data based on Ep1-Epp3
groups_aug = pd.util.hash_pandas_object(df_aug[MOD].round(6), index=False).astype(np.int64).to_numpy()
unique_groups_aug = np.unique(groups_aug)
print(f"Augmented data: {len(df_aug)} rows, {len(unique_groups_aug)} unique groups")

# Use GroupShuffleSplit (ensures samples from same group stay together)
pos_aug = np.arange(len(df_aug), dtype=int)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_pos_aug, va_pos_aug = next(gss.split(pos_aug, groups=groups_aug))

# Get training and validation sets
df_train_final = df_aug.iloc[tr_pos_aug].reset_index(drop=True)
df_val_final = df_aug.iloc[va_pos_aug].reset_index(drop=True)

# Check group distribution
groups_train = pd.util.hash_pandas_object(df_train_final[MOD].round(6), index=False).astype(np.int64)
groups_val = pd.util.hash_pandas_object(df_val_final[MOD].round(6), index=False).astype(np.int64)
unique_train = len(np.unique(groups_train))
unique_val = len(np.unique(groups_val))

print(f"\nSplit completed:")
print(f"  Training set: {len(df_train_final)} rows, {unique_train} groups")
print(f"  Validation set: {len(df_val_final)} rows, {unique_val} groups")
print(f"  Total: {unique_train + unique_val} groups")

# Verify no data leakage
overlap = set(np.unique(groups_train)) & set(np.unique(groups_val))
if overlap:
    print(f"  Warning: Found {len(overlap)} overlapping groups!")
else:
    print(f"  ✓ No data leakage: Training and validation groups are completely separate")

# === Refit scalers on augmented training set ===
print("\n[Scaler] Refitting scalers on augmented training set...")

sx_new = StandardScaler().fit(df_train_final[X_cols].to_numpy(float))
sy_new = StandardScaler().fit(df_train_final[y_cols].to_numpy(float))

# Save new scalers
os.makedirs("artifacts", exist_ok=True)
joblib.dump(sx_new, "artifacts/experimental/scaler_x_new.pkl")
joblib.dump(sy_new, "artifacts/experimental/scaler_y_new.pkl")
print("[Scaler] New scalers saved to artifacts/experimental/scaler_x_new.pkl & artifacts/experimental/scaler_y_new.pkl")

# Display scaler parameter comparison
print("\n=== Scaler Parameter Comparison ===")
print("Original scaler (based on full original data):")
for i, col in enumerate(X_cols):
    print(f"  {col}: mean={sx.mean_[i]:.4f}, std={sx.scale_[i]:.4f}")

print("\nNew scaler (based on augmented training set):")
for i, col in enumerate(X_cols):
    print(f"  {col}: mean={sx_new.mean_[i]:.4f}, std={sx_new.scale_[i]:.4f}")

print("\nOutput variables:")
print("Original sy:", [f"{col}: mean={sy.mean_[i]:.4f}, std={sy.scale_[i]:.4f}" for i, col in enumerate(y_cols)])
print("New sy_new:", [f"{col}: mean={sy_new.mean_[i]:.4f}, std={sy_new.scale_[i]:.4f}" for i, col in enumerate(y_cols)])

# === Save training and validation sets (with 6-tuple ordering) ===
def group_blocks_by_six_tuple(df, key_cols):
    """
    Re-order rows so that all rows for the same six-tuple are contiguous,
    preserving the order of first appearance.
    """
    uniq = df[key_cols].drop_duplicates(keep="first")
    blocks = []
    for _, row in uniq.iterrows():
        mask = (df[key_cols] == row.values).all(axis=1)
        blocks.append(df.loc[mask])
    if blocks:
        return pd.concat(blocks, axis=0, ignore_index=True)
    return df.copy()

os.makedirs("data", exist_ok=True)

# Sort data by 6-tuple groups before saving
print(f"\n[Sort] Sorting data by 6-tuple groups...")
df_train_sorted = group_blocks_by_six_tuple(df_train_final, MOD)
df_val_sorted = group_blocks_by_six_tuple(df_val_final, MOD)
df_aug_sorted = group_blocks_by_six_tuple(df_aug, MOD)

# Save sorted data
df_train_sorted.to_excel("data/combined_augmented_train.xlsx", index=False)
df_val_sorted.to_excel("data/val.xlsx", index=False)
df_aug_sorted.to_excel("data/combined_augmented_all_data.xlsx", index=False)

print(f"\n[Save] Data saved (sorted by 6-tuple groups):")
print(f"  - data/combined_augmented_train.xlsx (training set, {len(df_train_sorted)} rows)")
print(f"  - data/val.xlsx (validation set, {len(df_val_sorted)} rows)")
print(f"  - data/combined_augmented_all_data.xlsx (all augmented data, {len(df_aug_sorted)} rows)")

# === Analyze rare vs non-rare region data distribution ===
print(f"\n=== Rare vs Non-Rare Region Data Analysis ===")

# Define rare region bounds
RARE_BOUNDS_RAW = np.array([[1.20, 0.20], [5.80, 2.20]], dtype=np.float32)

def analyze_rare_vs_nonrare(df_data, set_name):
    """Analyze rare vs non-rare region data in a dataset"""
    
    # Get groups in this dataset (use precision 4 to match with original data)
    data_groups = pd.util.hash_pandas_object(df_data[MOD].round(4), index=False).astype(np.int64)
    unique_data_groups = np.unique(data_groups)
    
    # Classify groups as rare or non-rare based on original data
    rare_groups = []
    nonrare_groups = []
    
    for group_hash in unique_data_groups:
        # Find this group in original data to check if it contains rare samples
        orig_groups = pd.util.hash_pandas_object(df[MOD].round(4), index=False).astype(np.int64)
        orig_mask = orig_groups == group_hash
        
        if orig_mask.any():
            group_orig_data = df[orig_mask]
            
            # Check if this group has samples in rare region
            group_eph = group_orig_data['EpH'].values
            group_epph = group_orig_data['EppH'].values
            rare_mask = ((group_eph >= RARE_BOUNDS_RAW[0,0]) & (group_eph <= RARE_BOUNDS_RAW[1,0]) & 
                        (group_epph >= RARE_BOUNDS_RAW[0,1]) & (group_epph <= RARE_BOUNDS_RAW[1,1]))
            
            if np.any(rare_mask):
                rare_groups.append(group_hash)
            else:
                nonrare_groups.append(group_hash)
    
    # Count rows for each category
    rare_rows = 0
    nonrare_rows = 0
    
    for i, group_hash in enumerate(data_groups):
        if group_hash in rare_groups:
            rare_rows += 1
        elif group_hash in nonrare_groups:
            nonrare_rows += 1
    
    # Calculate original vs augmented for each category
    rare_original_rows = len(rare_groups) * 91  # Each original group has 91 rows
    nonrare_original_rows = len(nonrare_groups) * 91
    
    rare_augmented_rows = rare_rows - rare_original_rows
    nonrare_augmented_rows = nonrare_rows - nonrare_original_rows
    
    print(f"{set_name} ({len(df_data)} rows, {len(unique_data_groups)} groups):")
    print(f"  Rare region groups ({len(rare_groups)} groups, {rare_rows} rows):")
    print(f"    - Original data: {rare_original_rows} rows ({rare_original_rows/len(df_data)*100:.1f}% of total)")
    print(f"    - Augmented data: {rare_augmented_rows} rows ({rare_augmented_rows/len(df_data)*100:.1f}% of total)")
    if rare_original_rows > 0:
        print(f"    - Augmentation ratio: {rare_rows/rare_original_rows:.2f}x")
    else:
        print(f"    - Augmentation ratio: N/A (no rare groups)")
    
    print(f"  Non-rare region groups ({len(nonrare_groups)} groups, {nonrare_rows} rows):")
    print(f"    - Original data: {nonrare_original_rows} rows ({nonrare_original_rows/len(df_data)*100:.1f}% of total)")
    print(f"    - Augmented data: {nonrare_augmented_rows} rows ({nonrare_augmented_rows/len(df_data)*100:.1f}% of total)")
    if nonrare_original_rows > 0:
        print(f"    - Augmentation ratio: {nonrare_rows/nonrare_original_rows:.2f}x")
    else:
        print(f"    - Augmentation ratio: N/A (no non-rare groups)")
    
    return {
        'rare_groups': len(rare_groups),
        'nonrare_groups': len(nonrare_groups),
        'rare_rows': rare_rows,
        'nonrare_rows': nonrare_rows,
        'rare_original': rare_original_rows,
        'nonrare_original': nonrare_original_rows,
        'rare_augmented': rare_augmented_rows,
        'nonrare_augmented': nonrare_augmented_rows
    }

# Analyze each dataset
train_stats = analyze_rare_vs_nonrare(df_train_sorted, "Training set")
print()
val_stats = analyze_rare_vs_nonrare(df_val_sorted, "Validation set")
print()
all_stats = analyze_rare_vs_nonrare(df_aug_sorted, "Total augmented dataset")

# Summary statistics
print(f"\n=== Summary ===")
print(f"Original dataset analysis:")
# Analyze original data to show baseline
orig_groups = pd.util.hash_pandas_object(df[MOD].round(4), index=False).astype(np.int64)
orig_unique_groups = np.unique(orig_groups)

orig_rare_groups = 0
orig_nonrare_groups = 0

for group_hash in orig_unique_groups:
    orig_mask = orig_groups == group_hash
    group_data = df[orig_mask]
    
    group_eph = group_data['EpH'].values
    group_epph = group_data['EppH'].values
    rare_mask = ((group_eph >= RARE_BOUNDS_RAW[0,0]) & (group_eph <= RARE_BOUNDS_RAW[1,0]) & 
                (group_epph >= RARE_BOUNDS_RAW[0,1]) & (group_epph <= RARE_BOUNDS_RAW[1,1]))
    
    if np.any(rare_mask):
        orig_rare_groups += 1
    else:
        orig_nonrare_groups += 1

print(f"  - Rare region groups: {orig_rare_groups}/{len(orig_unique_groups)} ({orig_rare_groups/len(orig_unique_groups)*100:.1f}%)")
print(f"  - Non-rare region groups: {orig_nonrare_groups}/{len(orig_unique_groups)} ({orig_nonrare_groups/len(orig_unique_groups)*100:.1f}%)")
print(f"  - Only rare region groups were augmented (non-rare groups remain at 1.00x)")

#### 5. Training Process Visualization


In [ ]:
# Save training history
df_hist = pd.DataFrame(hist)
df_hist.to_csv("artifacts/experimental/rnn_loss_history.csv", index=False)
print("Training history saved to rnn_loss_history.csv")

# Moving average smoothing function
def moving_avg(x, k=5):
    """Calculate moving average"""
    if k <= 1:
        return np.array(x, dtype=float)
    x = np.array(x, dtype=float)
    pad = k//2
    xpad = np.pad(x, (pad, pad), mode="edge")
    kernel = np.ones(k) / k
    return np.convolve(xpad, kernel, mode="valid")

# Calculate smoothed losses
train_smooth = moving_avg(df_hist["train_loss"], k=5)
val_smooth = moving_avg(df_hist["val_loss"], k=5)

In [ ]:
# ===== Offline Plotting: Load Training History from CSV =====
# This cell reads the saved training history and plots loss curves
# Useful when you want to visualize results without retraining

import os

if os.path.exists("artifacts/experimental/rnn_loss_history.csv"):
    print("Loading saved training history from CSV...")
    df_hist_loaded = pd.read_csv("artifacts/experimental/rnn_loss_history.csv")
    
    # Moving average smoothing function
    def moving_avg(x, k=5):
        """Calculate moving average for smoothing"""
        if k <= 1:
            return np.array(x, dtype=float)
        x = np.array(x, dtype=float)
        pad = k // 2
        xpad = np.pad(x, (pad, pad), mode="edge")
        kernel = np.ones(k) / k
        return np.convolve(xpad, kernel, mode="valid")
    
    # Apply smoothing
    train_smooth = moving_avg(df_hist_loaded["train_loss"], k=5)
    val_smooth = moving_avg(df_hist_loaded["val_loss"], k=5)
    
    # Plot 1: Training and Validation Loss
    plt.figure(figsize=(8, 6))
    plt.plot(df_hist_loaded["epoch"], train_smooth, label="Train Loss", linewidth=4, color='blue')
    plt.plot(df_hist_loaded["epoch"], val_smooth, label="Validation Loss", linewidth=4, color='red')
    
    plt.xlabel('Epoch', fontsize=26, fontweight='bold', fontfamily='serif', labelpad=10)
    plt.ylabel('MSE Loss', fontsize=26, fontweight='bold', fontfamily='serif', labelpad=6)
    plt.xticks(fontsize=20, fontweight='bold', fontfamily='serif')
    plt.yticks(fontsize=20, fontweight='bold', fontfamily='serif')
    plt.legend(prop={'size': 20, 'weight': 'bold', 'family': 'serif'})
    
    ax = plt.gca()
    ax.xaxis.set_major_locator(MultipleLocator(base=20))
    ax.yaxis.set_major_locator(MultipleLocator(base=0.2))
    
    for spine in ax.spines.values():
        spine.set_linewidth(3)
    
    plt.tick_params(axis='both', which='major', length=8, width=3)
    plt.tight_layout()
    plt.show()
    
    # Plot 2: Learning Rate Schedule
    plt.figure(figsize=(8, 6))
    plt.plot(df_hist_loaded["epoch"], np.array(df_hist_loaded["lr"]) * 1000, linewidth=4, color='green')
    plt.xlabel('Epoch', fontsize=26, fontweight='bold', fontfamily='serif', labelpad=10)
    plt.ylabel(r'LR ($\times 10^{-3}$)', fontsize=26, fontweight='bold', fontfamily='serif', labelpad=6)
    plt.xticks(fontsize=20, fontweight='bold', fontfamily='serif')
    plt.yticks(fontsize=20, fontweight='bold', fontfamily='serif')
    
    ax = plt.gca()
    ax.xaxis.set_major_locator(MultipleLocator(base=20))
    ax.yaxis.set_major_locator(MultipleLocator(base=0.5))
    
    for spine in ax.spines.values():
        spine.set_linewidth(3)
    
    plt.tick_params(axis='both', which='major', length=8, width=3)
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"\nTraining Summary:")
    print(f"  Total epochs: {len(df_hist_loaded)}")
    print(f"  Best validation loss: {df_hist_loaded['val_loss'].min():.6f}")
    print(f"  Best epoch: {df_hist_loaded['val_loss'].idxmin() + 1}")
    print(f"  Final train loss: {df_hist_loaded['train_loss'].iloc[-1]:.6f}")
    print(f"  Final validation loss: {df_hist_loaded['val_loss'].iloc[-1]:.6f}")
else:
    print("Training history file not found: artifacts/experimental/rnn_loss_history.csv")
    print("Please run the training code first to generate the training history file.")

#### 6. Original vs Augmented Data Comparison


In [ ]:
# ===== Offline loading and plotting original vs augmented data comparison =====
# If saved data files exist, directly load and visualize comparison plots

import os

# Check if files exist
original_data_path = "data/TrainingData.xlsx"
augmented_data_path = "data/combined_augmented_all_data.xlsx"

if os.path.exists(original_data_path) and os.path.exists(augmented_data_path):
    print("Loading original and augmented data...")
    
    # Load original data
    df_original = pd.read_excel(original_data_path)
    print(f"Original data: {len(df_original)} rows")
    
    # Load augmented data
    df_augmented = pd.read_excel(augmented_data_path)
    print(f"Augmented data: {len(df_augmented)} rows")
    
    # Ensure required columns exist
    required_cols = ['EpH', 'EppH']
    if all(col in df_original.columns for col in required_cols) and all(col in df_augmented.columns for col in required_cols):
        
        # Plot EpH density comparison
        print("\nPlotting EpH density comparison...")
        data_o = df_original['EpH'].dropna().values
        data_a = df_augmented['EpH'].dropna().values
        
        kde_o = gaussian_kde(data_o)
        kde_a = gaussian_kde(data_a)
        
        x_min = min(data_o.min(), data_a.min())
        x_max = max(data_o.max(), data_a.max())
        x_vals = np.linspace(x_min, x_max, 500)
        
        plt.figure(figsize=(8, 8))
        plt.plot(x_vals, kde_o(x_vals), color='red', linewidth=3, label='Original')
        plt.plot(x_vals, kde_a(x_vals), color='blue', linewidth=3, label='Augmented')
        plt.fill_between(x_vals, kde_o(x_vals), alpha=0.3, color='red')
        plt.fill_between(x_vals, kde_a(x_vals), alpha=0.3, color='blue')
        
        plt.xlabel(r"$E_H^{\prime}$ (MPa)", fontsize=24, fontweight='bold', fontfamily='serif', labelpad=10)
        plt.ylabel("Dist.Density", fontsize=24, fontweight='bold', fontfamily='serif', labelpad=8)
        plt.xticks(fontsize=20, fontweight='bold', fontfamily='serif')
        plt.yticks(fontsize=20, fontweight='bold', fontfamily='serif')
        plt.legend(prop={'size': 20, 'weight': 'bold', 'family': 'serif'})
        
        ax = plt.gca()
        ax.xaxis.set_major_locator(MultipleLocator(base=1))
        ax.yaxis.set_major_locator(MultipleLocator(base=0.1))
        
        for spine in ax.spines.values():
            spine.set_linewidth(3)
        plt.tick_params(axis='both', which='major', length=8, width=3)
        plt.tight_layout()
        plt.show()
        
        # Plot EppH density comparison
        print("Plotting EppH density comparison...")
        data_o = df_original['EppH'].dropna().values
        data_a = df_augmented['EppH'].dropna().values
        
        kde_o = gaussian_kde(data_o)
        kde_a = gaussian_kde(data_a)
        
        x_min = min(data_o.min(), data_a.min())
        x_max = max(data_o.max(), data_a.max())
        x_vals = np.linspace(x_min, x_max, 500)
        
        plt.figure(figsize=(8, 8))
        plt.plot(x_vals, kde_o(x_vals), color='red', linewidth=3, label='Original')
        plt.plot(x_vals, kde_a(x_vals), color='blue', linewidth=3, label='Augmented')
        plt.fill_between(x_vals, kde_o(x_vals), alpha=0.3, color='red')
        plt.fill_between(x_vals, kde_a(x_vals), alpha=0.3, color='blue')
        
        plt.xlabel(r"$E_H^{\prime\prime}$ (MPa)", fontsize=24, fontweight='bold', fontfamily='serif', labelpad=10)
        plt.ylabel("Dist.Density", fontsize=24, fontweight='bold', fontfamily='serif', labelpad=8)
        plt.xticks(fontsize=20, fontweight='bold', fontfamily='serif')
        plt.yticks([0, 0.4, 0.8, 1.2], fontsize=20, fontweight='bold', fontfamily='serif')
        plt.legend(prop={'size': 20, 'weight': 'bold', 'family': 'serif'})
        
        ax = plt.gca()
        for spine in ax.spines.values():
            spine.set_linewidth(3)
        plt.tick_params(axis='both', which='major', length=8, width=3)
        plt.tight_layout()
        plt.show()
        
        # Plot scatter comparison
        print("Plotting scatter comparison...")
        
        # Sample data to reduce plotting points
        sample_size_orig = min(2000, len(df_original))
        sample_size_aug = min(8000, len(df_augmented))
        
        orig_sample = df_original.sample(n=sample_size_orig, random_state=42)
        aug_sample = df_augmented.sample(n=sample_size_aug, random_state=42)
        
        x_orig = orig_sample['EpH']
        y_orig = orig_sample['EppH']
        x_aug = aug_sample['EpH']
        y_aug = aug_sample['EppH']
        
        plt.figure(figsize=(16, 8))
        plt.scatter(x_orig, y_orig, s=6, alpha=0.6, c='red', label='Original')
        plt.scatter(x_aug, y_aug, s=6, alpha=0.4, c='blue', label='Augmented')
        
        plt.xlabel(r"$E_H^{\prime}$ (MPa)", fontsize=24, fontweight='bold', fontfamily='serif', labelpad=10)
        plt.ylabel(r"$E_H^{\prime\prime}$ (MPa)", fontsize=24, fontweight='bold', fontfamily='serif', labelpad=7)
        plt.xticks(fontsize=20, fontweight='bold', fontfamily='serif')
        plt.yticks(fontsize=20, fontweight='bold', fontfamily='serif')
        plt.legend(prop={'size': 20, 'weight': 'bold', 'family': 'serif'})
        
        ax = plt.gca()
        ax.xaxis.set_major_locator(MultipleLocator(base=1))
        ax.yaxis.set_major_locator(MultipleLocator(base=0.5))
        
        for spine in ax.spines.values():
            spine.set_linewidth(3)
        plt.tick_params(axis='both', which='major', length=8, width=3)
        plt.tight_layout()
        plt.show()
        
        print(f"\nData statistics:")
        print(f"Original data sample size: {sample_size_orig}")
        print(f"Augmented data sample size: {sample_size_aug}")
        print(f"Data augmentation ratio: {len(df_augmented)/len(df_original):.2f}x")
    else:
        print(f"Error: Data files missing required columns {required_cols}")
else:
    print("Data files not found:")
    if not os.path.exists(original_data_path):
        print(f"  - {original_data_path}")
    if not os.path.exists(augmented_data_path):
        print(f"  - {augmented_data_path}")
    print("Please run data augmentation code first to generate data files")

#### 7. Save Augmented Data


In [ ]:
# === Save model and generate summary ===
print("=== Saved Files Summary ===")

# Save model to artifacts folder
torch.save(model_rnn.state_dict(), "artifacts/experimental/best_bilstm.pt")
torch.save(model_rnn.state_dict(), "artifacts/experimental/rnn_model_new.pt")
print("\nModel files (artifacts folder):")
print("  - artifacts/experimental/best_bilstm.pt (RNN model)")
print("  - artifacts/experimental/rnn_model_new.pt (RNN model)")

# Save original scalers (for consistency, already saved earlier)
import pickle
with open("artifacts/experimental/scaler_x.pkl", "wb") as f:
    pickle.dump(sx, f)
with open("artifacts/experimental/scaler_y.pkl", "wb") as f:
    pickle.dump(sy, f)

print("\nScaler files (artifacts folder):")
print("  - artifacts/experimental/scaler_x.pkl (input scaler based on original data)")
print("  - artifacts/experimental/scaler_y.pkl (output scaler based on original data)")
print("  - artifacts/experimental/scaler_x_new.pkl (input scaler based on augmented training set - recommended)")
print("  - artifacts/experimental/scaler_y_new.pkl (output scaler based on augmented training set - recommended)")

print("\nData files (data folder):")
print(f"  - data/combined_augmented_train.xlsx (training set, {len(df_train_final)} rows, {len(np.unique(pd.util.hash_pandas_object(df_train_final[MOD].round(6), index=False)))} groups)")
print(f"  - data/val.xlsx (validation set, {len(df_val_final)} rows, {len(np.unique(pd.util.hash_pandas_object(df_val_final[MOD].round(6), index=False)))} groups)")
print(f"  - data/combined_augmented_all_data.xlsx (all augmented data, {len(df_aug)} rows, {len(np.unique(pd.util.hash_pandas_object(df_aug[MOD].round(6), index=False)))} groups)")

print("\n=== Data Augmentation Summary ===")
print(f"Original dataset: {len(df)} rows, {len(np.unique(pd.util.hash_pandas_object(df[MOD].round(6), index=False)))} groups")
print(f"Augmented dataset: {len(df_aug)} rows ({len(df_aug)/len(df):.2f}x), {len(np.unique(pd.util.hash_pandas_object(df_aug[MOD].round(6), index=False)))} groups")
print(f"Synthetic samples added: {len(df_aug) - len(df)} rows")
print(f"\nTraining set: {len(df_train_final)} rows ({len(df_train_final)/len(df_aug)*100:.1f}%), {len(np.unique(pd.util.hash_pandas_object(df_train_final[MOD].round(6), index=False)))} groups")
print(f"Validation set: {len(df_val_final)} rows ({len(df_val_final)/len(df_aug)*100:.1f}%), {len(np.unique(pd.util.hash_pandas_object(df_val_final[MOD].round(6), index=False)))} groups")

print("\nNote:")
print("  - Original scalers used for RNN training")
print("  - New scalers (recommended) reflect augmented data distribution for downstream models (e.g., diffusion)")
print("  - GroupShuffleSplit ensures no data leakage between training and validation sets")